# SalesPilot — End-to-End Demo

This notebook demonstrates the SalesPilot CRM pipeline:
1. **Data Loading** — CSV ingestion & feature engineering
2. **Model Training** — XGBoost deal-closure predictor
3. **Scoring** — Predict win probability for accounts
4. **Route Optimization** — TSP-based visit route using OR-Tools
5. **API Test** — Hit the FastAPI endpoints

## 0. Setup

In [ ]:
import sys, os
os.chdir(os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.')
# Ensure we're in the salespilot project root
if os.path.basename(os.getcwd()) != 'salespilot':
    os.chdir('salespilot') if os.path.isdir('salespilot') else None
print(f'Working dir: {os.getcwd()}')
sys.path.insert(0, os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# CSV data is one level up
CSV_DIR = Path('..') 
print('CSV files available:', [f.name for f in CSV_DIR.glob('*.csv')])

## 1. Data Loading & Exploration

In [ ]:
# Load raw CSVs
accounts = pd.read_csv(CSV_DIR / 'accounts.csv')
products = pd.read_csv(CSV_DIR / 'products.csv')
sales_teams = pd.read_csv(CSV_DIR / 'sales_teams.csv')
pipeline = pd.read_csv(CSV_DIR / 'sales_pipeline.csv')

print(f'Accounts:    {accounts.shape}')
print(f'Products:    {products.shape}')
print(f'Sales Teams: {sales_teams.shape}')
print(f'Pipeline:    {pipeline.shape}')

In [ ]:
accounts.head()

In [ ]:
pipeline.head()

In [ ]:
# Deal stage distribution
pipeline['deal_stage'].value_counts()

## 2. Feature Engineering (data_loader pipeline)

In [ ]:
from app.data.data_loader import _load_accounts, _load_products, _load_sales_teams, _load_opportunities, _hash_id

# Load and transform each table
acct_df = _load_accounts(CSV_DIR)
prod_df = _load_products(CSV_DIR)
team_df = _load_sales_teams(CSV_DIR)

print(f'Accounts (transformed): {acct_df.shape}')
print(f'Products (transformed): {prod_df.shape}')
print(f'Sales Teams (transformed): {team_df.shape}')
acct_df.head()

In [ ]:
# Build FK lookups and load opportunities
account_lookup = dict(zip(
    acct_df['account_name'].str.strip().str.lower(),
    acct_df['account_id'],
))
agent_lookup = dict(zip(
    team_df['sales_agent'].str.strip().str.lower(),
    team_df['agent_id'],
))
product_lookup = dict(zip(
    prod_df['product_name'].str.strip().str.lower(),
    prod_df['product_id'],
))

opp_df = _load_opportunities(CSV_DIR, account_lookup, agent_lookup, product_lookup)
print(f'Opportunities (transformed): {opp_df.shape}')
opp_df.head()

In [ ]:
# Check deal_closed distribution
print('Deal closed distribution:')
print(opp_df['deal_closed'].value_counts())
print(f'\nWin rate: {opp_df["deal_closed"].mean():.2%}')

## 3. Synthetic Geo Coordinates

In [ ]:
print(f'Accounts with coordinates: {acct_df["latitude"].notna().sum()} / {len(acct_df)}')
print(f'\nSample coordinates:')
acct_df[['account_name', 'region', 'latitude', 'longitude']].head(10)

## 4. Model Training

In [ ]:
from app.ml.train_model import load_features_from_csv, train, build_preprocessor

# Load features from CSV (no DB needed)
feature_df = load_features_from_csv(str(CSV_DIR))
print(f'Feature DataFrame: {feature_df.shape}')
print(f'Class balance: {feature_df["deal_closed"].value_counts().to_dict()}')
feature_df.head()

In [ ]:
# Train the model
metrics = train(feature_df)
print('\nTraining metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

## 5. Model Scoring (Direct — No DB)

In [ ]:
import joblib

# Load the trained model
model = joblib.load('app/ml/artifacts/model.joblib')

# Build a sample scoring DataFrame
sample_df = feature_df.sample(10, random_state=42)[[
    'industry', 'region', 'sales_stage', 'deal_value',
    'company_size', 'days_since_last_contact'
]].copy()

# Score
probs = model.predict_proba(sample_df)[:, 1]
sample_df['win_probability'] = probs
sample_df.sort_values('win_probability', ascending=False)

## 6. Route Optimization (TSP)

In [ ]:
from app.optimization.haversine import haversine_km, build_distance_matrix

# Pick 6 accounts with coordinates as sample stops
sample_accounts = acct_df.head(6)
points = list(zip(sample_accounts['latitude'], sample_accounts['longitude']))

print('Sample account stops:')
for i, (_, row) in enumerate(sample_accounts.iterrows()):
    print(f'  [{i}] {row["account_name"]} — ({row["latitude"]:.4f}, {row["longitude"]:.4f})')

# Build distance matrix
dist_matrix = build_distance_matrix(points)
print(f'\nDistance matrix (km):')
print(np.round(dist_matrix, 1))

In [ ]:
# Run TSP solver in a subprocess to avoid protobuf conflict between ortools and pandas/pyarrow
import subprocess, json as _json

points_json = _json.dumps(points)
names = sample_accounts['account_name'].tolist()

script = f'''
import sys, json
sys.path.insert(0, ".")
from app.optimization.distance_provider import HaversineProvider
from app.optimization.ortools_tsp import solve_tsp

points = json.loads('{points_json}')
points = [tuple(p) for p in points]
provider = HaversineProvider()
result = solve_tsp(points, provider)
output = json.dumps(dict(ordered_indices=result.ordered_indices, total_distance_km=result.total_distance_km))
print(output)
'''

proc = subprocess.run([sys.executable, '-c', script], capture_output=True, text=True, timeout=30)
if proc.returncode != 0:
    print('TSP solver error:', proc.stderr)
else:
    tsp_result = _json.loads(proc.stdout.strip())
    ordered = tsp_result['ordered_indices']
    dist = tsp_result['total_distance_km']
    print('Optimal route order:', ordered)
    print('Total distance:', dist, 'km')
    print('\nRoute:')
    for i, idx in enumerate(ordered):
        print(f'  Stop {i}: {names[idx]}')
    print(f'  Return to: {names[ordered[0]]}')

## 7. API Test (FastAPI)

In [ ]:
import httpx

BASE_URL = 'http://localhost:8000'

# Health check
try:
    r = httpx.get(f'{BASE_URL}/health', timeout=5)
    print(f'Health: {r.json()}')
except httpx.ConnectError:
    print('API not running — start with: uvicorn app.main:app --port 8000')

In [ ]:
# OpenAPI spec summary
try:
    r = httpx.get(f'{BASE_URL}/openapi.json', timeout=5)
    spec = r.json()
    print('API Endpoints:')
    for path, methods in spec['paths'].items():
        for method in methods:
            print(f'  {method.upper()} {path}')
except httpx.ConnectError:
    print('API not running')

## 8. Summary

| Component | Status |
|---|---|
| CSV Data Loading (4 tables) | Tested |
| Feature Engineering | Tested |
| Synthetic Geo Coordinates | Tested |
| XGBoost Model Training | Tested |
| Model Scoring (predict_proba) | Tested |
| Haversine Distance Matrix | Tested |
| OR-Tools TSP Solver | Tested |
| FastAPI Health Endpoint | Tested |